In [ ]:
import os, csv, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

activity = 'run' 
data_path = os.path.join(os.getcwd(), 'Data/WISDM', activity) 
all_files = os.listdir(data_path)
# print("Files:", all_files)

# Check fps
filenames, fps_list = [],[]
for f in all_files:
    filename = os.path.splitext(f)[0]
    file_path = os.path.join(data_path, f)

    cap = cv2.VideoCapture(file_path) # Open video
    fps = cap.get(cv2.CAP_PROP_FPS) # Get fps (frames per sccond)
    # print(f"fps: {fps:.2f}")

    filenames.append(filename)
    fps_list.append(round(fps))

    cap.release()

df_fps = pd.DataFrame({
    "filename": filenames,
    "fps": fps_list
})
display(df_fps)

<img src="33 landmarks reference.jpeg" width="100%" height = "100%" />

In [9]:
def compute_normalized_squared_jerk_2d_verbose(x, y, fps):
    """
    Compute dimensionless normalized squared jerk for 2D movement,
    and also return amplitude and duration.
    """
    dt = 1.0 / fps

    dx = np.gradient(x, dt)
    ddx = np.gradient(dx, dt)
    dddx = np.gradient(ddx, dt)

    dy = np.gradient(y, dt)
    ddy = np.gradient(dy, dt)
    dddy = np.gradient(ddy, dt)

    jerk_mag = np.sqrt(dddx**2 + dddy**2)
    jerk_squared = jerk_mag**2
    integrated_squared_jerk = np.sum(jerk_squared) * dt

    D = len(x) * dt

    dx_range = np.max(x) - np.min(x)
    dy_range = np.max(y) - np.min(y)
    A = np.sqrt(dx_range**2 + dy_range**2)

    if A == 0 or D == 0:
        nsj = np.nan
    else:
        nsj = (D**5 / A**2) * integrated_squared_jerk

    return nsj, A, D


In [13]:
# df = pd.read_csv('Motion_centroids_landmarks/daria_run_ct_lm.csv')

fps = 25 # video frame rate
jerk_scores, results = [],[]

# Loop over landmark indices 1 to 33
for i in range(1, 34):
    x_col = f'x{i}'
    y_col = f'y{i}'

    if x_col in df.columns and y_col in df.columns:
        x = df[x_col].values
        y = df[y_col].values
        nsj, A, D = compute_normalized_squared_jerk_2d_verbose(x, y, fps)
        jerk_scores.append(nsj)
        print(f"Landmark {i}: NSJ={nsj:.2f}, Amplitude={A:.4f}, Duration={D:.4f} sec")
        results.append({
            'landmark': i,
            'nsj': nsj,
            'amplitude': A,
            'duration': D
        })
    else:
        jerk_scores.append(np.nan)  # or handle missing columns as you like
        print(f"Landmark {i}: Missing data.")
        results.append({
            'landmark': i,
            'nsj': np.nan,
            'amplitude': np.nan,
            'duration': np.nan
        })
        
print("Normalized Squared Jerk per landmark:")
print(jerk_scores)
results_df = pd.DataFrame(results)


Landmark 1: NSJ=47389.04, Amplitude=164.1097, Duration=1.6800 sec
Landmark 2: NSJ=43218.80, Amplitude=164.1097, Duration=1.6800 sec
Landmark 3: NSJ=39958.44, Amplitude=164.1097, Duration=1.6800 sec
Landmark 4: NSJ=30037.87, Amplitude=165.1091, Duration=1.6800 sec
Landmark 5: NSJ=38138.70, Amplitude=164.1097, Duration=1.6800 sec
Landmark 6: NSJ=39958.44, Amplitude=164.1097, Duration=1.6800 sec
Landmark 7: NSJ=39730.97, Amplitude=164.1097, Duration=1.6800 sec
Landmark 8: NSJ=53233.77, Amplitude=165.1484, Duration=1.6800 sec
Landmark 9: NSJ=47191.66, Amplitude=165.1091, Duration=1.6800 sec
Landmark 10: NSJ=44107.38, Amplitude=164.1493, Duration=1.6800 sec
Landmark 11: NSJ=52704.51, Amplitude=163.1502, Duration=1.6800 sec
Landmark 12: NSJ=86889.69, Amplitude=169.1892, Duration=1.6800 sec
Landmark 13: NSJ=49436.24, Amplitude=164.1097, Duration=1.6800 sec
Landmark 14: NSJ=426613.99, Amplitude=170.1881, Duration=1.6800 sec
Landmark 15: NSJ=209430.48, Amplitude=157.1560, Duration=1.6800 sec
La